In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

In [2]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [4]:
log_reg = LogisticRegression(C=1, solver='liblinear', random_state=42)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print("Logistic Regression classification report:\n", classification_report(y_test, y_pred))

Logistic Regression classification report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



In [5]:
rf_clf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print("Random Forest classification report:\n", classification_report(y_test, y_pred_rf))

Random Forest classification report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print("XGBoost classification report:\n", classification_report(y_test, y_pred_xgb))

XGBoost classification report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



In [7]:
from imblearn.combine import SMOTETomek
smote_tomek = SMOTETomek(random_state=42)
X_train_res, y_train_res = smote_tomek.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [8]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb_res = xgb_clf.predict(X_test)
print("XGBoost classification report after SMOTE-Tomek:\n", classification_report(y_test, y_pred_xgb_res))

XGBoost classification report after SMOTE-Tomek:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



### Track Experiments Using ML Flow

In [9]:
models =[
    (
        "LogisticRegression", 
        {"C": 1, "solver": "liblinear", "random_state": 42},
        LogisticRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "RandomForestClassifier", 
        {"n_estimators": 50, "max_depth": 3, "random_state": 42},
        RandomForestClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier", 
        {"use_label_encoder": False, "eval_metric": "logloss", "random_state": 42},
        XGBClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier with SMOTE", 
        {"use_label_encoder": False, "eval_metric": "logloss", "random_state": 42},
        XGBClassifier(),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [10]:
reports=[]

for model_name, params, model, train_set, test_set in models:
    X_train_model, y_train_model = train_set
    X_test_model, y_test_model = test_set

    model.set_params(**params)
    model.fit(X_train_model, y_train_model)
    
    y_pred_model = model.predict(X_test_model)
    report_dict = classification_report(y_test_model, y_pred_model, output_dict=True)
    reports.append((model_name, report_dict))

In [11]:
reports

[('LogisticRegression',
  {'0': {'precision': 0.9454545454545454,
    'recall': 0.9629629629629629,
    'f1-score': 0.9541284403669725,
    'support': 270.0},
   '1': {'precision': 0.6,
    'recall': 0.5,
    'f1-score': 0.5454545454545454,
    'support': 30.0},
   'accuracy': 0.9166666666666666,
   'macro avg': {'precision': 0.7727272727272727,
    'recall': 0.7314814814814814,
    'f1-score': 0.749791492910759,
    'support': 300.0},
   'weighted avg': {'precision': 0.9109090909090909,
    'recall': 0.9166666666666666,
    'f1-score': 0.91326105087573,
    'support': 300.0}}),
 ('RandomForestClassifier',
  {'0': {'precision': 0.9676258992805755,
    'recall': 0.9962962962962963,
    'f1-score': 0.9817518248175182,
    'support': 270.0},
   '1': {'precision': 0.9545454545454546,
    'recall': 0.7,
    'f1-score': 0.8076923076923077,
    'support': 30.0},
   'accuracy': 0.9666666666666667,
   'macro avg': {'precision': 0.961085676913015,
    'recall': 0.8481481481481481,
    'f1-score'

In [12]:
import mlflow

### Run in command line "mlflow ui" command

In [13]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Anomaly Detection Experiment_02")

for i, element in enumerate(models):
    model_name, params, model, train_set, test_set = element
    report_dict = reports[i][1]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        for param_name, param_value in params.items():
            mlflow.log_param(param_name, param_value)
        mlflow.log_metric("precision_class_1", report_dict['1']['precision'])
        mlflow.log_metric("recall_class_1", report_dict['1']['recall'])
        mlflow.log_metric("f1_score_class_1", report_dict['1']['f1-score'])
        mlflow.log_metric("precision_class_0", report_dict['0']['precision'])
        mlflow.log_metric("recall_class_0", report_dict['0']['recall'])
        mlflow.log_metric("f1_score_class_0", report_dict['0']['f1-score'])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")

        else:
            mlflow.sklearn.log_model(model, "model")

2026/07/15 07:36:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://localhost:5000/#/experiments/4/runs/49dcc202b63f40ffb01a21d7886982bd
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/07/15 07:36:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/15 07:36:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://localhost:5000/#/experiments/4/runs/8a6f968e0eec4ca49a4ba249b558d281
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/07/15 07:37:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/4/runs/d9f5db00a3c94b38bd21d61ed34f0ff3
🧪 View experiment at: http://localhost:5000/#/experiments/4
🏃 View run XGBClassifier with SMOTE at: http://localhost:5000/#/experiments/4/runs/59358df696e14167bb584994c5b4d62e
🧪 View experiment at: http://localhost:5000/#/experiments/4


### Register the model


In [14]:
model_name = 'XGBClassifier with SMOTE'
run_id = "e8f989d5d23344bdb5203047a1b25e7a"
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri=model_uri, name=model_name)
result.version

Registered model 'XGBClassifier with SMOTE' already exists. Creating a new version of this model...
2026/07/15 07:37:09 WARNING mlflow.tracking._model_registry.fluent: Run with id e8f989d5d23344bdb5203047a1b25e7a has no artifacts at artifact path 'model', registering model based on models:/m-f103d8f768f247bf965d199dd737ed14 instead
2026/07/15 07:37:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier with SMOTE, version 3
Created version '3' of model 'XGBClassifier with SMOTE'.


'3'

In [15]:
model_version = 1
model_uri = f"models:/{model_name}@challenger"
loaded_model = mlflow.xgboost.load_model(model_uri=model_uri)

y_pred_loaded_model = loaded_model.predict(X_test)
print("Classification report for the loaded model:\n", classification_report(y_test, y_pred_loaded_model))

Classification report for the loaded model:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [16]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = 'anomaly-detection-prod'

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '2' of model 'XGBClassifier with SMOTE' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1784081232409, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1784081232409, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='e8f989d5d23344bdb5203047a1b25e7a', run_link='', source='models:/XGBClassifier with SMOTE/2', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [17]:

model_uri = f"models:/{prod_model}@champion"
loaded_model = mlflow.xgboost.load_model(model_uri=model_uri)

y_pred_loaded_model = loaded_model.predict(X_test)
print("Classification report for the loaded model:\n", classification_report(y_test, y_pred_loaded_model))

Classification report for the loaded model:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300

